In [68]:
import os
import gc
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch import nn

In [ ]:
args = {
    'experiment_name': 'test_with_shuffled_data',
    'seed': 20,
    'region': "BE",
    'use_gpu': False,
    'predict_horizon': 24,
    'recalibration_shift_days': 200,
    'val_ratio': 0.2, # % of the train data to use for validation
    'save_checkpoints': False,
    'save_tensorboard': False,


    # Data parameters
    'use_scaled_data': True,  # if True, load df_full_scaled.csv; if False, use df_full.csv
    'target_scaled': 'TARG__target_scaled',  # target column name in the scaled dataset
    'target': 'TARG__target',
    'train_start': '2019-01-01',
    'train_end': '2023-10-01',
    'test_start': '2023-10-01', # included
    'test_end': '2024-10-01', # esluded

    # DNN parameters
    'target_quantiles': [i/100 for i in range(1, 100)],
    'perform_RevIN': True, # only true allowed for now
    'hidden_size': 128,  
    'hidden_layers': 2,
    'activation_fn': 'ReLU', # currently only ReLU is allowed
    'dropout_rate': 0.1,
    'context_window_days': [-1,-2,-7], # days indexes before the forecast horizon
    'full_history_hours': 168, # total hours to use for RevIN stats
    'epochs': 800,
    'patience': 20,
    'learning_rate': 5e-4,
    'batch_size': 128,
    'num_workers': 4,
}

In [70]:
if torch.cuda.is_available():
    device = torch.device('cuda:0')
    print('Using GPU')
else:
    device = torch.device('cpu')
    print('Using CPU')

# Create checkpoint and log folders
os.makedirs(f'log_dir', exist_ok=True)
os.makedirs(f'checkpoints', exist_ok=True)


Using CPU


In [71]:
# LOADING DATA
target_col = args['target_scaled']
denorm_params = pd.read_csv("./BE/df_target_denorm_params.csv")
denorm_params['date'] = pd.to_datetime(denorm_params['date'])

df_raw = pd.read_csv("./BE/df_full_scaled.csv")
feature_cols = list(df_raw.columns)
feature_cols.remove(target_col)
feature_cols.remove('date')
df_raw['date'] = pd.to_datetime(df_raw['date'])

# Store the effective target column for use in the rest of the pipeline
effective_target = target_col

cons_cols = [col for col in feature_cols if 'CONS' in col]
futu_cols = [col for col in feature_cols if 'CONS' not in col]
args['input_data_shape'] = len(args['context_window_days'])*24 + (len(futu_cols)*args['predict_horizon'] + len(cons_cols))

In [72]:
class DNN(nn.Module):
    def __init__(self, args):
        super(DNN, self).__init__()
        self.args = args

        self.input_layer = nn.Sequential(
            nn.Linear(self.args['input_data_shape'], self.args['hidden_size']), 
            nn.ReLU(),
            nn.Dropout(p=self.args['dropout_rate'])
        )

        self.hidden_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.args['hidden_size'], self.args['hidden_size']),
                nn.ReLU(),
                nn.Dropout(p=self.args['dropout_rate']),
            ) for _ in range(self.args['hidden_layers'] - 1)
        ])

        self.out_features = len(self.args['target_quantiles'])
        self.output_layer = nn.Linear(self.args['hidden_size'], self.out_features*self.args['predict_horizon'])

    def forward(self, past_target, future_features, target_mean, target_std): 
        # past_target: (batch, len(context_window_days)*24)
        # future_features: (batch, num_normal*24 + num_cons)
        
        # Inputs are already normalized by the dataset
        x = torch.cat([past_target, future_features], dim=1) # (batch, total_input_features)

        assert x.shape[1] == self.args['input_data_shape'], f"Input shape mismatch: expected {self.args['input_data_shape']}, got {x.shape[1]}"

        x = self.input_layer(x) 

        for layer in self.hidden_layers:
            x = layer(x)
        out = self.output_layer(x)

        # Reshape to (batch, horizon, quantiles)
        out = out.view(-1, self.args['predict_horizon'], self.out_features)
        
        # Denormalize + reshape for broadcasting
        out = out * target_std.view(-1, 1, 1) + target_mean.view(-1, 1, 1)
        
        # fix quantiles crossing
        out, _ = torch.sort(out, dim=-1) 
        return out 


In [73]:
class PinballLoss(torch.nn.Module):
    def __init__(self, quantiles: list):
        super().__init__()
        self.quantiles = quantiles

    def forward(self, y_true, y_pred):
        # y_true: (batch, horizon)
        # y_pred: (batch, horizon, quantiles)
        
        # Ensure quantiles are on the same device as input
        if not hasattr(self, 'q_tensor') or self.q_tensor.device != y_true.device:
            self.q_tensor = torch.tensor(self.quantiles, dtype=y_true.dtype, device=y_true.device).view(1, 1, -1)
            
        # Broadcast y_true to (batch, horizon, 1) to match y_pred's quantile dim
        error = y_true.unsqueeze(-1) - y_pred
        
        # Vectorized Pinball Loss: max(q * error, (q - 1) * error)
        # q_tensor broadcasts to (1, 1, n_quantiles)
        loss = torch.mean(torch.maximum(self.q_tensor * error, (self.q_tensor - 1) * error))
        
        return loss

    def get_config(self):
        return {
            "num_quantiles": self.quantiles,
        }


In [74]:
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, df: pd.DataFrame, feature_cols: list, target: str, context_window_days: list, full_history_hours: int, prediction_horizon: int, daily_aligned: bool = False):
        self.df = df.reset_index(drop=True)
        self.feature_cols = feature_cols
        self.target = target
        self.context_window_days = context_window_days
        self.full_history_hours = full_history_hours
        self.prediction_horizon = prediction_horizon
        self.all_features_unnorm = self.df[self.feature_cols].values.astype(np.float32)
        self.all_target_unnorm = self.df[self.target].values.astype(np.float32)
        self.dates = self.df['date'].values
        self.static_features_idx = [self.feature_cols.index(col) for col in self.feature_cols if 'CONS' in col]
        self.dynamic_features_idx = [self.feature_cols.index(col) for col in self.feature_cols if 'CONS' not in col]

        # Pre-compute valid indices: only those where prediction starts at midnight
        total_window = self.full_history_hours + self.prediction_horizon
        all_count = max(0, len(self.df) - total_window + 1)
        if daily_aligned:
            self.valid_indices = [
                i for i in range(all_count)
                if pd.to_datetime(self.dates[i + self.full_history_hours]).hour == 0
            ]
        else:
            self.valid_indices = list(range(all_count))


    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        """
        history window: [idx : idx + full_history_hours]
        prediction horizon: [idx + full_history_hours : idx + full_history_hours + prediction_horizon]

        - Normalization stats are computed on the past window of lenght: [full_history_hours] from settings
        - Normalization also affects the time encoding variables
        """
        # Map external idx to internal data index
        idx = self.valid_indices[idx]
        past_target_window_unnorm = self.all_target_unnorm[idx : idx + self.full_history_hours]
        past_features_window_unnorm = self.all_features_unnorm[idx : idx + self.full_history_hours]
        
        # Calculate RevIN stats
        past_target_mean = past_target_window_unnorm.mean()
        past_target_std = past_target_window_unnorm.std() + 1e-5
        past_features_mean = past_features_window_unnorm.mean(axis=0)
        past_features_std = past_features_window_unnorm.std(axis=0) + 1e-5
        
        # Normalize target for the full window
        full_window_target_unnorm = self.all_target_unnorm[idx : idx + self.full_history_hours + self.prediction_horizon]
        full_window_target_norm = (full_window_target_unnorm - past_target_mean) / past_target_std
        
        # Normalize all features
        full_window_features_norm = (self.all_features_unnorm[idx : idx + self.full_history_hours + self.prediction_horizon] - past_features_mean) / past_features_std
        
        # 1. Past Target: select specific days from the normalized history
        past_target_input_list = []
        for day_offset in self.context_window_days:
            start_in_window = self.full_history_hours + (day_offset * 24)
            end_in_window = start_in_window + 24
            past_target_input_list.append(full_window_target_norm[start_in_window:end_in_window])
        
        past_target_input_norm = np.concatenate(past_target_input_list) # (len(days)*24,)
        
        # 2. Future Features: prediction horizon part of full_window_features_norm
        future_features_window_norm = full_window_features_norm[self.full_history_hours : self.full_history_hours + self.prediction_horizon]
        
        # Dynamic features (all 24h)
        future_dynamic_features_norm = future_features_window_norm[:, self.dynamic_features_idx].flatten() 
        # Static features (only 1st value)
        future_static_features_norm = future_features_window_norm[0, self.static_features_idx]
        
        future_features_input_norm = np.concatenate([future_dynamic_features_norm, future_static_features_norm])
        
        # 3. Future Target (raw/unnormalized)
        future_target_label_unnorm = self.all_target_unnorm[idx + self.full_history_hours : idx + self.full_history_hours + self.prediction_horizon]

        return (torch.from_numpy(past_target_input_norm), 
                torch.from_numpy(future_features_input_norm), 
                torch.from_numpy(future_target_label_unnorm), 
                torch.tensor(past_target_mean, dtype=torch.float32), 
                torch.tensor(past_target_std, dtype=torch.float32))

In [75]:
# TRAINING FUNCTION

def train_on_split(train_raw, suffix, feature_cols):
    best_val_loss = float('inf')
    model = DNN(args)
    model.to(device)
    pinball_loss = PinballLoss(args['target_quantiles']).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args['learning_rate'])

    # TensorBoard writer for this specific ricalibration
    # Using suffix (which contains region, seed and week) to separate logs
    log_dir = os.path.join(f'log_dir', 'tensorboard', suffix)
    writer = SummaryWriter(log_dir=log_dir)

    # Build ONE dataset from all train data, with daily alignment (1 sample per day)
    full_dataset = CustomDataset(
        train_raw,
        feature_cols, 
        effective_target,
        context_window_days=args['context_window_days'],
        full_history_hours=args['full_history_hours'],
        prediction_horizon=args['predict_horizon'],
        daily_aligned=True  # always daily-aligned
    )
    
    n_samples = len(full_dataset)
    n_val = int(n_samples * args['val_ratio'])
    n_train = n_samples - n_val

    # Shuffle sample indices BEFORE splitting (like the paper)
    all_indices = list(range(n_samples))
    np.random.shuffle(all_indices)
    train_indices = all_indices[:n_train]
    val_indices = all_indices[n_train:]

    print(f"Train/Val split: {n_train} train samples, {n_val} val samples (total: {n_samples})")

    train_subset = torch.utils.data.Subset(full_dataset, train_indices)
    val_subset = torch.utils.data.Subset(full_dataset, val_indices)

    train_loader = DataLoader(
        train_subset, 
        batch_size=args['batch_size'], 
        shuffle=True, 
        num_workers=args['num_workers'], 
        pin_memory=True, 
        persistent_workers=False
    )

    val_loader = DataLoader(
        val_subset, 
        batch_size=args['batch_size'], 
        num_workers=args['num_workers'], 
        pin_memory=True, 
        persistent_workers=False
    )

    patience_counter = 0
    for epoch in range(args['epochs']):
        # train
        model.train()
        train_losses = []
        for (past_target, future_features, future_target, target_mean, target_std) in train_loader:
            past_target = past_target.to(device)
            future_features = future_features.to(device)
            future_target = future_target.to(device)
            target_mean = target_mean.to(device)
            target_std = target_std.to(device)
            optimizer.zero_grad()
            output = model(past_target, future_features, target_mean, target_std)
            loss = pinball_loss(future_target, output)
            train_losses.append(loss.item())
            loss.backward()
            optimizer.step()

        # validation 
        model.eval()
        val_losses = [] 
        with torch.no_grad():
            for (past_target, future_features, future_target, target_mean, target_std) in val_loader:
                past_target = past_target.to(device)
                future_features = future_features.to(device)
                future_target = future_target.to(device)
                target_mean = target_mean.to(device)
                target_std = target_std.to(device)
                output = model(past_target, future_features, target_mean, target_std)
                loss = pinball_loss(future_target, output)
                val_losses.append(loss.item())

        print(f"Epoch {epoch+1} - Train Loss: {np.mean(train_losses):.4f} - Val Loss: {np.mean(val_losses):.4f}")
        
        # Log to TensorBoard
        writer.add_scalar('Loss/train', np.mean(train_losses), epoch)
        writer.add_scalar('Loss/val', np.mean(val_losses), epoch)

        if np.mean(val_losses) < best_val_loss: 
            print(f"New best validation loss: {np.mean(val_losses):.4f}, saving model...")
            best_val_loss = np.mean(val_losses)
            torch.save(model.state_dict(), f'checkpoints/best_model_{suffix}.pth')
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter > 0: print(f"Patience counter: {patience_counter} out of {args['patience']}")
        if patience_counter >= args['patience']:
            print(f"Patience reached, stopping training...")
            break
    
    # Explicitly delete loaders and datasets to trigger worker shutdown and memory release
    writer.close()
    del train_loader, val_loader, train_subset, val_subset, full_dataset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [76]:
# WALK FORWARD EXPERIMENT

train_raw = df_raw[(df_raw['date'] >= pd.to_datetime(args['train_start'])) & (df_raw['date'] < pd.to_datetime(args['train_end']))]
test_raw = df_raw[(df_raw['date'] >= pd.to_datetime(args['test_start'])) & (df_raw['date'] < pd.to_datetime(args['test_end']))]

print(f"Loaded data from {df_raw['date'].iloc[0]} to {df_raw['date'].iloc[-1]}")

day_count = 0 
while len(test_raw) > 0: 
    print(f"\n{'='*60}\nDay {day_count}: {test_raw['date'].iloc[0].strftime('%Y-%m-%d')}")

    # Check if we need to retrain
    if day_count % args['recalibration_shift_days'] == 0:
        print(f"Retraining - MarketRegion: {args['region']} - Seed: {args['seed']} - Retrain split: {day_count // args['recalibration_shift_days']} / {int(365/args['recalibration_shift_days'])}")
        print("First day of train data: ", train_raw['date'].iloc[0])
        print("Last day of train data: ", train_raw['date'].iloc[-1])
        print("First day of test data: ", test_raw['date'].iloc[0])
        print("Last day of test data: ", test_raw['date'].iloc[-1])
        print(f"Train data shape: {train_raw.shape}")
        print(f"Test data shape: {test_raw.shape}")
        
        train_on_split(
            train_raw=train_raw,
            suffix=f"week{day_count // args['recalibration_shift_days']}_{args['region']}_{args['seed']}",
            feature_cols=feature_cols
        )

    # Evaluation
    model = DNN(args)
    model.to(device)
    model.load_state_dict(torch.load(f"checkpoints/best_model_week{day_count // args['recalibration_shift_days']}_{args['region']}_{args['seed']}.pth"))
    model.eval()

    test_dataset = CustomDataset(
        pd.concat([train_raw, test_raw.iloc[:args['predict_horizon']]]),
        feature_cols,
        effective_target,
        context_window_days=args['context_window_days'],
        full_history_hours=args['full_history_hours'],
        prediction_horizon=args['predict_horizon'],
    )
    # We want the last sample of the test_dataset which corresponds to predicting the current test_raw horizon
    past_target, future_features, future_target, target_mean, target_std = test_dataset[len(test_dataset)-1]

    print("First sample of test interval: ", test_dataset.df['date'].iloc[test_dataset.valid_indices[-1] + test_dataset.full_history_hours])
    print("Last sample of test interval: ", test_dataset.df['date'].iloc[test_dataset.valid_indices[-1] + test_dataset.full_history_hours + test_dataset.prediction_horizon - 1])
    assert test_dataset.df['date'].iloc[test_dataset.valid_indices[-1] + test_dataset.full_history_hours] == test_raw['date'].iloc[0], "First sample of test interval does not match first sample of test_raw"
    assert test_dataset.df['date'].iloc[test_dataset.valid_indices[-1] + test_dataset.full_history_hours + test_dataset.prediction_horizon - 1] == test_raw['date'].iloc[args['predict_horizon'] - 1], "Last sample of test interval does not match last sample of test_raw"

    with torch.no_grad():
        past_target = past_target.unsqueeze(0).to(device)
        future_features = future_features.unsqueeze(0).to(device)
        future_target = future_target.to(device)
        target_mean = target_mean.to(device)
        target_std = target_std.to(device)
        output = model(past_target, future_features, target_mean, target_std).squeeze(0) # (horizon, quantiles)

    # If we have less than a full horizon left in test_raw, 
    # we should only take the entries that correspond to the test_raw entries.
    L = min(len(test_raw), args['predict_horizon'])
    if L < args['predict_horizon']:
        output = output[:L] # take the first L entries (corresponding to the start of the horizon)
        future_target = future_target[:L]

    # Denormalize if using scaled data
    if args.get('use_scaled_data', False) and denorm_params is not None:
        test_dates = test_raw['date'].iloc[:L].values
        # Ensure strict alignment by date
        denorm_slice = denorm_params.set_index('date').loc[test_dates]
        p0 = torch.tensor(denorm_slice['CONS__TARG__target_trasf_p0'].values, dtype=output.dtype).to(device)
        p1 = torch.tensor(denorm_slice['CONS__TARG__target_trasf_p1'].values, dtype=output.dtype).to(device)
        # output shape: (L, num_quantiles) -> denormalize each row
        output = output * p1.unsqueeze(1) + p0.unsqueeze(1)
        future_target = future_target * p1 + p0

    # compute the metrics (just mae for now)
    median_idx = args['target_quantiles'].index(0.5) 
    mae = torch.mean(torch.abs(output[:, median_idx] - future_target)).item()
    print(f"MAE: {mae}")

    # SHIFT DATASETS
    train_raw = pd.concat([train_raw, test_raw.iloc[:args['predict_horizon']]]) # concatenate the first args['predict_horizon'] days of test to train
    test_raw = test_raw.iloc[args['predict_horizon']:] # remove the first args['predict_horizon'] days of test from test
    train_raw = train_raw.iloc[args['predict_horizon']:] # remove the first args['predict_horizon'] days of train from train
    day_count += 1

Loaded data from 2018-12-25 00:00:00 to 2025-04-17 23:00:00

Day 0: 2023-10-01
Retraining - MarketRegion: BE - Seed: 20 - Retrain split: 0 / 0
First day of train data:  2019-01-01 00:00:00
Last day of train data:  2023-09-30 23:00:00
First day of test data:  2023-10-01 00:00:00
Last day of test data:  2024-09-30 23:00:00
Train data shape: (41616, 8)
Test data shape: (8784, 8)
Train/Val split: 1382 train samples, 345 val samples (total: 1727)


/home/saltysoup/dnn_price_forecasting/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch 1 - Train Loss: 0.1155 - Val Loss: 0.0998
New best validation loss: 0.0998, saving model...
Epoch 2 - Train Loss: 0.0905 - Val Loss: 0.0852
New best validation loss: 0.0852, saving model...
Epoch 3 - Train Loss: 0.0824 - Val Loss: 0.0755
New best validation loss: 0.0755, saving model...
Epoch 4 - Train Loss: 0.0720 - Val Loss: 0.0682
New best validation loss: 0.0682, saving model...
Epoch 5 - Train Loss: 0.0655 - Val Loss: 0.0636
New best validation loss: 0.0636, saving model...
Epoch 6 - Train Loss: 0.0605 - Val Loss: 0.0583
New best validation loss: 0.0583, saving model...
Epoch 7 - Train Loss: 0.0572 - Val Loss: 0.0558
New best validation loss: 0.0558, saving model...
Epoch 8 - Train Loss: 0.0547 - Val Loss: 0.0547
New best validation loss: 0.0547, saving model...
Epoch 9 - Train Loss: 0.0537 - Val Loss: 0.0541
New best validation loss: 0.0541, saving model...
Epoch 10 - Train Loss: 0.0526 - Val Loss: 0.0530
New best validation loss: 0.0530, saving model...
Epoch 11 - Train Lo